# Hallmark Python Demo: init, info, add, commit, checkout, status, and clone


## Setup


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from hallmark import Repo
from hallmark.downloader import download_remote_data
from hallmark.error import DestinationExistsError


In [2]:
%%bash
rm -rf repo1
rm -rf hallmark-demo-clone
mkdir repo1


## 1. Initialize the hallmark repo


In [3]:
repo1 = Repo.init("repo1")
repo1


Repo(state=State(config={'data': [{'encoding': None}], 'remote': None}, meta={}, data=Empty DataFrame
Columns: [sha1]
Index: []), dothm=<hallmark.dothm.Dothm '/Users/ramadithyamuthukumarasamy/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo/repo1/.hm/.git'>, worktree=Worktree('/Users/ramadithyamuthukumarasamy/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo/repo1'))

In [4]:
%%bash
echo "=== .hm directory ==="
ls repo1/.hm/
echo ""
echo "=== initial config.yml ==="
cat repo1/.hm/config.yml
echo ""
echo "=== objects stored so far ==="
find repo1/.hm/objects -type f 2>/dev/null | wc -l


=== .hm directory ===
config.yml
data.tsv
meta.yml
README.md

=== initial config.yml ===
# Edit this file only if your branch needs regex substitutions or a preset remote.
# For simple names, you can just run: hallmark add "a{a}_i{i}.h5"
data:
  -
    # fmt: "{release}_{source}_{year}_{doy:03d}_{band}.uvfits"
    encoding:
      # aspin: m([0-9]+(\.[0-9]+)?|\.[0-9]+)
remote:
  # name: origin
  # url: https://example.com/path/to/data/

=== objects stored so far ===
       0


### Repository info

The Python API exposes the same local hallmark paths directly on the `Repo` object.


In [5]:
print(f'dot-hallmark repo: "{repo1.dothm.path}"')
print(f'hallmark worktree: "{repo1.worktree}"')


dot-hallmark repo: "/Users/ramadithyamuthukumarasamy/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo/repo1/.hm"
hallmark worktree: "/Users/ramadithyamuthukumarasamy/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo/repo1"


## 2. Create data files on `main`


In [6]:
%%bash
pushd repo1
for f in a{0,0.75}_i{0,30,60,90,120}.h5; do echo "$f" > "$f"; done
echo "Files in repo1/:"
ls *.h5
popd


~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo/repo1 ~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo
Files in repo1/:
a0_i0.h5
a0_i120.h5
a0_i30.h5
a0_i60.h5
a0_i90.h5
a0.75_i0.h5
a0.75_i120.h5
a0.75_i30.h5
a0.75_i60.h5
a0.75_i90.h5
~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo


## 3. Add files and commit on `main`


In [7]:
pf = repo1.add("a{a}_i{i}.h5")
pf


,path,a,i
0,a0.75_i0.h5,0.75,0.0
1,a0.75_i120.h5,0.75,120.0
2,a0.75_i30.h5,0.75,30.0
3,a0.75_i60.h5,0.75,60.0
4,a0.75_i90.h5,0.75,90.0
5,a0_i0.h5,0.00,0.0
6,a0_i120.h5,0.00,120.0
7,a0_i30.h5,0.00,30.0
8,a0_i60.h5,0.00,60.0
9,a0_i90.h5,0.00,90.0


In [8]:
%%bash
echo "=== config.yml after add ==="
cat repo1/.hm/config.yml
echo ""
echo "=== data.tsv after add ==="
cat repo1/.hm/data.tsv


=== config.yml after add ===
data:
- fmt: a{a}_i{i}.h5
  encoding: null
remote: null

=== data.tsv after add ===
sha1	a	i
f656d419104e7af783c119186a4d81b15563310f	0.75	0
ec4dae9f0b8f69ac42f0c290fc84e0d6f1bec6cd	0.75	120
890b2ea509575be9bc74ad515731a2562cd7406e	0.75	30
d9cfcdd0b44d81c10f88e126d5b1eea4914259a0	0.75	60
ed2cfc6bb157a42729d3c6f45b228b432edec4be	0.75	90
18682593a012503b71935017dbc044665254a23c	0	0
1c057a4885fd04379ee1217375720665713c902f	0	120
829b87d845b367a9cc87df6ff510d1bdb7444aca	0	30
075a30e0a9c4dddd92447f463219d4c2a842883d	0	60
de40f4983d86342b03809bfcc09958f43f359b89	0	90


In [9]:
repo1.commit("add main dataset")


True

In [10]:
%%bash
echo "Objects stored after commit:"
find repo1/.hm/objects -type f | wc -l


Objects stored after commit:
      10


### Commit log


In [ ]:
print(repo1.log())


### Clean status


In [11]:
repo1.status()


{'branch': 'main',
 'staged': {'added': [], 'modified': [], 'deleted': []},
 'worktree': {'modified': [], 'deleted': []},
 'untracked': []}

### Current add modes

- `repo.add("a{a}_i{i}.h5")` parses files using the branch fmt and updates `config.yml`.
- `repo.add(".")` rebuilds `data.tsv` from files that currently exist using the fmt already set in `config.yml`.
- `repo.set_config(fmt=...)` is the explicit way to change branch fmt before `repo.add(".")`.


## 4. Demo: `repo.add(".")` removes deleted files from the manifest

We use a temporary branch so the main workflow stays intact.
On this branch we switch to `b*` files first so it is easy to distinguish from `main`.
Because the filename family changes, we update the branch fmt explicitly with `repo.set_config(...)` before using `repo.add(".")`.


In [12]:
repo1.checkout("sync-demo")


True

In [13]:
%%bash
pushd repo1
rm a*.h5
for f in b{0,0.75}_i{0,30,60,90,120}.h5; do echo "$f" > "$f"; done
echo "Files after switching sync-demo to b files:"
ls *.h5
popd


~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo/repo1 ~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo
Files after switching sync-demo to b files:
b0_i0.h5
b0_i120.h5
b0_i30.h5
b0_i60.h5
b0_i90.h5
b0.75_i0.h5
b0.75_i120.h5
b0.75_i30.h5
b0.75_i60.h5
b0.75_i90.h5
~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo


In [14]:
repo1.set_config(fmt="b{a}_i{i}.h5")
repo1.state.config


{'data': [{'fmt': 'b{a}_i{i}.h5', 'encoding': None}], 'remote': None}

In [15]:
%%bash
pushd repo1
echo "config.yml after repo.set_config:"
cat .hm/config.yml
popd


~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo/repo1 ~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo
config.yml after repo.set_config:
data:
- fmt: b{a}_i{i}.h5
  encoding: null
remote: null
~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo


In [16]:
%%bash
pushd repo1
rm b0.75_i{0,30,60,90,120}.h5
echo "Files after deleting some tracked b files:"
ls *.h5
popd


~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo/repo1 ~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo
Files after deleting some tracked b files:
b0_i0.h5
b0_i120.h5
b0_i30.h5
b0_i60.h5
b0_i90.h5
~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo


In [17]:
repo1.add(".")


,path,a,i
0,b0_i0.h5,0.0,0.0
1,b0_i120.h5,0.0,120.0
2,b0_i30.h5,0.0,30.0
3,b0_i60.h5,0.0,60.0
4,b0_i90.h5,0.0,90.0


In [18]:
%%bash
echo "Manifest after repo.add dot:"
cat repo1/.hm/data.tsv


Manifest after repo.add dot:
sha1	a	i
56676e3e5629923617f8c9e542bd1c2ccaf8fa3a	0	0
683e82f624b175c075ac3ea0cca56e94cd7422f8	0	120
cfdbae05e3afc29354d948a32263f7ac1ed5850b	0	30
3a947fb1280b5bca62615d6c7543b88080588e7a	0	60
1711d3b47eac6ae479b6f0f240f83f41b393ebb4	0	90


In [19]:
repo1.status()


{'branch': 'sync-demo',
 'staged': {'added': ['b0_i0.h5',
   'b0_i120.h5',
   'b0_i30.h5',
   'b0_i60.h5',
   'b0_i90.h5'],
  'modified': [],
  'deleted': ['a0.75_i0.h5',
   'a0.75_i120.h5',
   'a0.75_i30.h5',
   'a0.75_i60.h5',
   'a0.75_i90.h5',
   'a0_i0.h5',
   'a0_i120.h5',
   'a0_i30.h5',
   'a0_i60.h5',
   'a0_i90.h5']},
 'worktree': {'modified': [], 'deleted': []},
 'untracked': []}

In [20]:
repo1.commit("sync manifest with hallmark add dot")


True

In [21]:
repo1.checkout("main")


True

In [22]:
%%bash
echo "Files after checkout back to main:"
ls repo1/*.h5


Files after checkout back to main:
repo1/a0_i0.h5
repo1/a0_i120.h5
repo1/a0_i30.h5
repo1/a0_i60.h5
repo1/a0_i90.h5
repo1/a0.75_i0.h5
repo1/a0.75_i120.h5
repo1/a0.75_i30.h5
repo1/a0.75_i60.h5
repo1/a0.75_i90.h5


## 5. Checkout a new branch

The new branch inherits the same fmt from `main`.


In [23]:
repo1.checkout("experiment")


True

In [24]:
%%bash
echo "Current .hm branch:"
git -C repo1/.hm branch --show-current
echo ""
echo "config.yml on experiment:"
cat repo1/.hm/config.yml
echo ""
echo "Files after checkout:"
ls repo1/*.h5


Current .hm branch:
experiment

config.yml on experiment:
data:
- fmt: a{a}_i{i}.h5
  encoding: null
remote: null

Files after checkout:
repo1/a0_i0.h5
repo1/a0_i120.h5
repo1/a0_i30.h5
repo1/a0_i60.h5
repo1/a0_i90.h5
repo1/a0.75_i0.h5
repo1/a0.75_i120.h5
repo1/a0.75_i30.h5
repo1/a0.75_i60.h5
repo1/a0.75_i90.h5


## 6. Replace the experiment branch dataset with new files that still match the same fmt


In [25]:
%%bash
pushd repo1
rm a*.h5
for f in a{1,1.5}_i{15,45,75,105,110}.h5; do echo "$f" > "$f"; done
echo "Files after replacing the dataset on experiment:"
ls *.h5
popd


~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo/repo1 ~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo
Files after replacing the dataset on experiment:
a1_i105.h5
a1_i110.h5
a1_i15.h5
a1_i45.h5
a1_i75.h5
a1.5_i105.h5
a1.5_i110.h5
a1.5_i15.h5
a1.5_i45.h5
a1.5_i75.h5
~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo


In [26]:
repo1.add(".")


,path,a,i
0,a1.5_i105.h5,1.5,105.0
1,a1.5_i110.h5,1.5,110.0
2,a1.5_i15.h5,1.5,15.0
3,a1.5_i45.h5,1.5,45.0
4,a1.5_i75.h5,1.5,75.0
5,a1_i105.h5,1.0,105.0
6,a1_i110.h5,1.0,110.0
7,a1_i15.h5,1.0,15.0
8,a1_i45.h5,1.0,45.0
9,a1_i75.h5,1.0,75.0


In [27]:
repo1.status()


{'branch': 'experiment',
 'staged': {'added': ['a1.5_i105.h5',
   'a1.5_i110.h5',
   'a1.5_i15.h5',
   'a1.5_i45.h5',
   'a1.5_i75.h5',
   'a1_i105.h5',
   'a1_i110.h5',
   'a1_i15.h5',
   'a1_i45.h5',
   'a1_i75.h5'],
  'modified': [],
  'deleted': ['a0.75_i0.h5',
   'a0.75_i120.h5',
   'a0.75_i30.h5',
   'a0.75_i60.h5',
   'a0.75_i90.h5',
   'a0_i0.h5',
   'a0_i120.h5',
   'a0_i30.h5',
   'a0_i60.h5',
   'a0_i90.h5']},
 'worktree': {'modified': [], 'deleted': []},
 'untracked': []}

In [28]:
%%bash
echo "experiment data.tsv before commit:"
cat repo1/.hm/data.tsv


experiment data.tsv before commit:
sha1	a	i
f0ca3ed468c8796645672d2ebb48d48ae49b6d3c	1.5	105
9d1801d46e071b98a87afcd1edc59fe9302b9a4b	1.5	110
ed34eda23a05ea4c3cd1bf41988c6e2f76cf41e1	1.5	15
b47e5f555daf30afea5b0f99cadad1e2d8c689ec	1.5	45
b2a40517fd5681088f88903165ae54af3c75cf25	1.5	75
635b4b44a00c0a2bde5a7f0e263bb98dcbdae6e2	1	105
24ef5885eff61f1a8fcd77c19329a2651eca0228	1	110
577954873e813a982f70867b49a374a230df3650	1	15
1580545be2a5d485338e3538c7b16709d5fa6e4a	1	45
e2406e4872637ef09f574731be5b5ba5e8aa5761	1	75


In [29]:
repo1.commit("replace experiment dataset")


True

In [30]:
%%bash
echo "Objects stored after experiment commit:"
find repo1/.hm/objects -type f | wc -l


Objects stored after experiment commit:
      25


## 7. Checkout back to `main`


In [31]:
repo1.checkout("main")


True

In [32]:
%%bash
echo "Files after checkout main:"
ls repo1/*.h5
echo ""
echo "main data.tsv:"
cat repo1/.hm/data.tsv


Files after checkout main:
repo1/a0_i0.h5
repo1/a0_i120.h5
repo1/a0_i30.h5
repo1/a0_i60.h5
repo1/a0_i90.h5
repo1/a0.75_i0.h5
repo1/a0.75_i120.h5
repo1/a0.75_i30.h5
repo1/a0.75_i60.h5
repo1/a0.75_i90.h5

main data.tsv:
sha1	a	i
f656d419104e7af783c119186a4d81b15563310f	0.75	0
ec4dae9f0b8f69ac42f0c290fc84e0d6f1bec6cd	0.75	120
890b2ea509575be9bc74ad515731a2562cd7406e	0.75	30
d9cfcdd0b44d81c10f88e126d5b1eea4914259a0	0.75	60
ed2cfc6bb157a42729d3c6f45b228b432edec4be	0.75	90
18682593a012503b71935017dbc044665254a23c	0	0
1c057a4885fd04379ee1217375720665713c902f	0	120
829b87d845b367a9cc87df6ff510d1bdb7444aca	0	30
075a30e0a9c4dddd92447f463219d4c2a842883d	0	60
de40f4983d86342b03809bfcc09958f43f359b89	0	90


## 8. Checkout back to `experiment` to verify


In [33]:
repo1.checkout("experiment")


True

In [34]:
%%bash
echo "Files after checkout experiment:"
ls repo1/*.h5
echo ""
echo "experiment data.tsv:"
cat repo1/.hm/data.tsv


Files after checkout experiment:
repo1/a1_i105.h5
repo1/a1_i110.h5
repo1/a1_i15.h5
repo1/a1_i45.h5
repo1/a1_i75.h5
repo1/a1.5_i105.h5
repo1/a1.5_i110.h5
repo1/a1.5_i15.h5
repo1/a1.5_i45.h5
repo1/a1.5_i75.h5

experiment data.tsv:
sha1	a	i
f0ca3ed468c8796645672d2ebb48d48ae49b6d3c	1.5	105
9d1801d46e071b98a87afcd1edc59fe9302b9a4b	1.5	110
ed34eda23a05ea4c3cd1bf41988c6e2f76cf41e1	1.5	15
b47e5f555daf30afea5b0f99cadad1e2d8c689ec	1.5	45
b2a40517fd5681088f88903165ae54af3c75cf25	1.5	75
635b4b44a00c0a2bde5a7f0e263bb98dcbdae6e2	1	105
24ef5885eff61f1a8fcd77c19329a2651eca0228	1	110
577954873e813a982f70867b49a374a230df3650	1	15
1580545be2a5d485338e3538c7b16709d5fa6e4a	1	45
e2406e4872637ef09f574731be5b5ba5e8aa5761	1	75


## 9. Uncommitted changes block checkout


In [35]:
%%bash
pushd repo1
echo "a2_i15.h5" > a2_i15.h5
echo "Files after creating a2_i15.h5:"
ls *.h5
popd


~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo/repo1 ~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo
Files after creating a2_i15.h5:
a1_i105.h5
a1_i110.h5
a1_i15.h5
a1_i45.h5
a1_i75.h5
a1.5_i105.h5
a1.5_i110.h5
a1.5_i15.h5
a1.5_i45.h5
a1.5_i75.h5
a2_i15.h5
~/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo


In [36]:
repo1.add(".")


,path,a,i
0,a1.5_i105.h5,1.5,105.0
1,a1.5_i110.h5,1.5,110.0
2,a1.5_i15.h5,1.5,15.0
3,a1.5_i45.h5,1.5,45.0
4,a1.5_i75.h5,1.5,75.0
5,a1_i105.h5,1.0,105.0
6,a1_i110.h5,1.0,110.0
7,a1_i15.h5,1.0,15.0
8,a1_i45.h5,1.0,45.0
9,a1_i75.h5,1.0,75.0


### Dirty status


In [37]:
repo1.status()


{'branch': 'experiment',
 'staged': {'added': ['a2_i15.h5'], 'modified': [], 'deleted': []},
 'worktree': {'modified': [], 'deleted': []},
 'untracked': []}

To fix this, commit the staged changes first or discard them.


## 10. Clone a hallmark repository

This clone demo matches the CLI notebook: it clones the hallmark repo, then downloads the configured remote data files. If you rerun the clone cell, the notebook catches `DestinationExistsError` and prints only Hallmark's fatal message.


In [38]:
clone_url = "https://github.com/l6a/hallmark-demo-repo.hm.git"
clone_path = Path("hallmark-demo-clone")

try:
    cloned_repo = Repo.clone(clone_url, clone_path)
    print(f'Successfully cloned to "{clone_path}"')
    print("Downloading remote data files...")
    results = download_remote_data(cloned_repo, clone_path, show_progress=True)
    if results["failed"] == 0:
        mb_total = results["total_bytes"] / (1024 * 1024)
        print(f'Successfully downloaded {results["succeeded"]} files ({mb_total:.1f} MB)')
    else:
        print(
            "Download completed with errors: "
            f'{results["succeeded"]} succeeded, {results["failed"]} failed'
        )
        for error in results["errors"]:
            print(f'  - {error}')
except DestinationExistsError as exc:
    print(exc)


Successfully cloned to "hallmark-demo-clone"


100%|██████████| 8/8 [00:00<00:00, 31.32file/s]

Successfully downloaded 8 files (4.3 MB)


In [39]:
cloned_repo = Repo(clone_path)
print(f'dot-hallmark repo: "{cloned_repo.dothm.path}"')
print(f'hallmark worktree: "{cloned_repo.worktree}"')


dot-hallmark repo: "/Users/ramadithyamuthukumarasamy/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo/hallmark-demo-clone/.hm"
hallmark worktree: "/Users/ramadithyamuthukumarasamy/Library/CloudStorage/GoogleDrive-ramadithya.research@gmail.com/My Drive/uni/research/github/hallmark/demo/hallmark-demo-clone"


In [40]:
%%bash
echo "=== cloned config.yml ==="
cat hallmark-demo-clone/.hm/config.yml


=== cloned config.yml ===
data:
  - fmt: '{release}_{source}_{year}_{doy:03d}_{band}_{pipeline}_{step}_{type}.uvfits'
remote:
  name: origin
  url: https://data.cyverse.org/dav-anon/iplant/commons/cyverse_curated/EHTC_FirstM87Results_Apr2019/uvfits/